In [1]:
from openai import OpenAI
client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

def get_embedding(text, model="text-embedding-nomic-embed-text-v1.5"):
   text = text.replace("\n", " ")
   return client.embeddings.create(input=[text], model=model).data[0].embedding


## Section 1: Make Embeddings

In [2]:
phrases = [
    "feline friends say",
    "meow",
    "canine companions say",
    "woof",
    "bovine buddies say",
    "moo",
    "a quarterback throws a football",
]

embeddings = []
for phrase in phrases:
    print(f"Embedding: '{phrase}'")
    embeddings.append(get_embedding(phrase))

print(f"\nDone! Each embedding has {len(embeddings[0])} dimensions.")


Embedding: 'feline friends say'
Embedding: 'meow'
Embedding: 'canine companions say'
Embedding: 'woof'
Embedding: 'bovine buddies say'
Embedding: 'moo'
Embedding: 'a quarterback throws a football'

Done! Each embedding has 768 dimensions.


## Section 2: Reduce to 2D with PCA

In [3]:
from sklearn.decomposition import PCA
import numpy as np

embedding_matrix = np.array(embeddings)

# Reduce to 2 dimensions using PCA
pca = PCA(n_components=2, random_state=42)
coords_2d = pca.fit_transform(embedding_matrix)

print("Original shape:", embedding_matrix.shape)
print("Reduced shape: ", coords_2d.shape)
print(f"Explained variance: {pca.explained_variance_ratio_.sum():.2%}")


Original shape: (7, 768)
Reduced shape:  (7, 2)
Explained variance: 51.33%


## Section 3: Visualize on an Interactive Scatter Plot

In [4]:
import plotly.express as px
import pandas as pd

df = pd.DataFrame({
    "x": coords_2d[:, 0],
    "y": coords_2d[:, 1],
    "phrase": phrases,
})

fig = px.scatter(
    df,
    x="x",
    y="y",
    text="phrase",
    hover_name="phrase",
    title="Semantic Map: Embeddings Reduced to 2D",
    width=800,
    height=600,
)

fig.update_traces(
    textposition="top center",
    marker=dict(size=12),
)

fig.update_layout(
    xaxis_title="Dimension 1",
    yaxis_title="Dimension 2",
    showlegend=False,
)

fig.show()
